# ERGT 01 - Attention Evidence Ladder

هدف این نوت بوک این است که قبل از اجرای بلند، زنجیره شواهد GeoAttention v2 را با یک budget کوتاه بسازد:

1. strict controls و observerها را دوباره تولید می کند.
2. memory و causal shortest-path geometry را چک می کند.
3. GeoAttention v2 را با کنترل های training کوتاه اجرا می کند.
4. در پایان یک gate ساده می دهد: pass، نیازمند اجرای بلندتر، یا نیازمند بررسی.

اصل کار این است: اول اندازه گیری و کنترل، بعد تزریق attention. این نوت بوک auxiliary loss، معماری کامل، reasoning و intelligence-space را عمدا وارد نمی کند.

## 0. Run Profile

`smoke` برای پیدا کردن سریع خطاهای شکل tensor، mask، NaN/Inf و logging است. بعد از پاس شدن smoke، `sanity` را اجرا کن. اگر آن هم سالم بود، `evidence` برای 1000 تا 1500 step مناسب است.

اجرای training فقط وقتی شروع می شود که دیتای آماده در `data/processed/wikitext2_gpt2_ctx256` وجود داشته باشد، مگر اینکه `PREPARE_DATA_IF_MISSING=True` شود.

In [ ]:
from pathlib import Path
import copy
import json
import math
import os
import subprocess
import sys
import time
import zipfile

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import torch
    DEFAULT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEFAULT_DEVICE = "cpu"

RUN_PROFILE = "smoke"  # "smoke", "sanity", or "evidence"
DEVICE = DEFAULT_DEVICE
SEED = 2027
ALPHA = 0.1

RUN_UNIT_TESTS = True
REGENERATE_OBSERVER_REPORTS = True
RUN_TRAINING = True
PREPARE_DATA_IF_MISSING = False
STOP_ON_FIRST_FAILURE = True
ABORT_ON_FAILED_OBSERVER_GATE = True
ABORT_ON_FAILED_ATTENTION_GATE = True
EXPORT_REPORT_BUNDLE = True
DOWNLOAD_BUNDLE_IN_COLAB = True
INCLUDE_INSTANTANEOUS_CONTROL = True
REPORT_BUNDLE_NAME = "ergt_01_attention_evidence_report_bundle.zip"

PROFILE_SETTINGS = {
    "smoke": {
        "max_steps": 20,
        "eval_interval": 10,
        "batch_size": 4,
        "n_layers": 2,
        "n_heads": 4,
        "hidden_dim": 128,
        "ffn_dim": 512,
        "warmup_steps": 4,
        "per_run_timeout_minutes": 12,
    },
    "sanity": {
        "max_steps": 200,
        "eval_interval": 50,
        "batch_size": 8,
        "n_layers": 4,
        "n_heads": 4,
        "hidden_dim": 256,
        "ffn_dim": 1024,
        "warmup_steps": 40,
        "per_run_timeout_minutes": 30,
    },
    "evidence": {
        "max_steps": 1200,
        "eval_interval": 100,
        "batch_size": 16,
        "n_layers": 6,
        "n_heads": 8,
        "hidden_dim": 512,
        "ffn_dim": 2048,
        "warmup_steps": 200,
        "per_run_timeout_minutes": 35,
    },
}

if RUN_PROFILE not in PROFILE_SETTINGS:
    raise ValueError(f"Unknown RUN_PROFILE: {RUN_PROFILE}")

BUDGET = PROFILE_SETTINGS[RUN_PROFILE]
RUN_ID = f"{time.strftime('%Y%m%d_%H%M%S')}_{RUN_PROFILE}"
RUN_ROOT = Path("runs/notebook_01_attention_evidence_ladder") / RUN_ID
CONFIG_DIR = RUN_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"project_root={PROJECT_ROOT}")
print(f"device={DEVICE}")
print(f"run_profile={RUN_PROFILE}")
print(f"run_root={RUN_ROOT}")
print(json.dumps(BUDGET, indent=2, sort_keys=True))

## 1. Utilities

In [ ]:
def read_json(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, sort_keys=True)
        f.write("\n")


def run_cmd(cmd, *, timeout_sec=None, check=True):
    start = time.perf_counter()
    print("\n$ " + " ".join(str(part) for part in cmd))
    try:
        completed = subprocess.run(
            [str(part) for part in cmd],
            cwd=PROJECT_ROOT,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            timeout=timeout_sec,
        )
    except subprocess.TimeoutExpired as exc:
        elapsed = time.perf_counter() - start
        output = exc.stdout or ""
        if isinstance(output, bytes):
            output = output.decode("utf-8", errors="replace")
        print(output[-12000:])
        print(f"TIMEOUT after {elapsed / 60:.1f} min")
        if check:
            raise
        return {"returncode": "timeout", "elapsed_seconds": elapsed, "stdout": output}

    elapsed = time.perf_counter() - start
    print(completed.stdout[-12000:])
    print(f"returncode={completed.returncode}, elapsed={elapsed / 60:.2f} min")
    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed with code {completed.returncode}: {cmd}")
    return {
        "returncode": completed.returncode,
        "elapsed_seconds": elapsed,
        "stdout": completed.stdout,
    }


def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    records = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            records.append(json.loads(line))
    return records


def finite_or_none(value):
    if value is None:
        return None
    try:
        value = float(value)
    except Exception:
        return None
    return value if math.isfinite(value) else None


def export_report_bundle(reason="manual"):
    bundle_path = RUN_ROOT / REPORT_BUNDLE_NAME
    include_paths = [
        Path("notebooks/ERGT_01_Attention_Evidence_Ladder.ipynb"),
        RUN_ROOT / "pretraining_gate_summary.json",
        RUN_ROOT / "attention_evidence_summary.json",
    ]

    if "CONFIG_DIR" in globals() and CONFIG_DIR.exists():
        include_paths.extend(sorted(CONFIG_DIR.glob("*.json")))

    if "report_paths" in globals():
        include_paths.extend(Path(path) for path in report_paths.values())

    if "run_plan" in globals():
        for item in run_plan:
            output_dir = Path(item["output_dir"])
            include_paths.extend(
                output_dir / name
                for name in [
                    "config.json",
                    "metrics.json",
                    "baseline_results.json",
                    "model_summary.json",
                    "train_log.jsonl",
                    "progress_log.jsonl",
                ]
            )

    filtered_paths = []
    seen = set()
    for path in include_paths:
        path = Path(path)
        if not path.exists() or not path.is_file():
            continue
        if "checkpoints" in path.parts or path.suffix.lower() in {".pt", ".pth", ".ckpt"}:
            continue
        resolved = path.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        filtered_paths.append(path)

    manifest_path = RUN_ROOT / "report_bundle_manifest.json"
    manifest = {
        "reason": reason,
        "bundle_name": REPORT_BUNDLE_NAME,
        "run_profile": RUN_PROFILE,
        "run_root": RUN_ROOT.as_posix(),
        "excluded": ["checkpoints/", "*.pt", "*.pth", "*.ckpt"],
        "files": [path.as_posix() for path in filtered_paths],
    }
    write_json(manifest_path, manifest)
    filtered_paths.append(manifest_path)

    bundle_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in filtered_paths:
            try:
                arcname = path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
            except ValueError:
                arcname = path.name
            archive.write(path, arcname)

    print(f"report_bundle={bundle_path}")
    print(f"files_in_bundle={len(filtered_paths)}")

    if DOWNLOAD_BUNDLE_IN_COLAB:
        try:
            from google.colab import files
            files.download(str(bundle_path))
        except Exception as exc:
            print(f"Colab auto-download skipped: {exc}")
    return bundle_path


def print_table(rows, columns):
    if not rows:
        print("<no rows>")
        return
    widths = []
    for col in columns:
        width = len(col)
        for row in rows:
            width = max(width, len(str(row.get(col, ""))))
        widths.append(width)
    header = " | ".join(col.ljust(width) for col, width in zip(columns, widths))
    print(header)
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(" | ".join(str(row.get(col, "")).ljust(width) for col, width in zip(columns, widths)))

## 2. Dataset Gate

این نوت بوک training را روی دیتای آماده اجرا می کند. اگر دیتا حاضر نباشد، بخش observerها همچنان قابل اجراست، ولی training controls skip می شوند.

In [ ]:
DATA_DIR = Path("data/processed/wikitext2_gpt2_ctx256")
DATA_READY = (DATA_DIR / "metadata.json").exists()

if not DATA_READY and PREPARE_DATA_IF_MISSING:
    run_cmd([
        sys.executable,
        "scripts/prepare_wikitext2.py",
        "--output-dir",
        DATA_DIR,
        "--tokenizer",
        "gpt2",
        "--context-length",
        "256",
    ], timeout_sec=60 * 60)
    DATA_READY = (DATA_DIR / "metadata.json").exists()

print(f"data_dir={DATA_DIR}")
print(f"data_ready={DATA_READY}")
if DATA_READY:
    print(json.dumps(read_json(DATA_DIR / "metadata.json"), indent=2, sort_keys=True)[:2000])
else:
    print("Training controls will be skipped. Set PREPARE_DATA_IF_MISSING=True or prepare data manually.")

## 3. Fast Unit Smoke

In [ ]:
if RUN_UNIT_TESTS:
    test_targets = [
        "tests/test_geo_attention.py",
        "tests/test_geoattention_v2_report.py",
        "tests/test_causal_shortest_path_geometry.py",
        "tests/test_relational_memory_observer.py",
        "tests/test_strict_w_controls.py",
    ]
    run_cmd([sys.executable, "-m", "pytest", *test_targets], timeout_sec=10 * 60)
else:
    print("Unit smoke disabled.")

## 4. Regenerate Measurement and Observer Reports

این بخش هنوز مدل را train نمی کند. فقط شواهد پایه را بازتولید می کند: کنترل های منصفانه، میدان رابطه ای، resonance، Phi، reconstruction، memory، causal geometry و smoke مکانیکی GeoAttention v2.

In [ ]:
observer_scripts = [
    "experiments/create_measurement_contract_report.py",
    "experiments/create_strict_w_controls_report.py",
    "experiments/create_relational_field_observer_report.py",
    "experiments/create_resonant_response_observer_report.py",
    "experiments/create_information_potential_report.py",
    "experiments/create_reconstruction_gate_report.py",
    "experiments/create_relational_memory_observer_report.py",
    "experiments/create_causal_shortest_path_geometry_report.py",
    "experiments/create_geoattention_v2_report.py",
]

if REGENERATE_OBSERVER_REPORTS:
    for script in observer_scripts:
        run_cmd([sys.executable, script], timeout_sec=10 * 60)
else:
    print("Observer regeneration disabled; existing reports will be read.")

In [ ]:
report_paths = {
    "measurement_contracts": "runs/contracts/measurement_contract_report.json",
    "strict_w_controls": "runs/contracts/strict_w_controls_report.json",
    "relational_field": "runs/observers/relational_field_observer_report.json",
    "resonant_response": "runs/observers/resonant_response_observer_report.json",
    "information_potential_phi": "runs/observers/information_potential_report.json",
    "reconstruction_gate": "runs/observers/reconstruction_gate_report.json",
    "relational_memory": "runs/observers/relational_memory_observer_report.json",
    "causal_shortest_path": "runs/observers/causal_shortest_path_geometry_report.json",
    "geoattention_v2_mechanics": "runs/geoattention_v2/geoattention_v2_report.json",
}

report_rows = []
for name, path in report_paths.items():
    path = Path(path)
    if not path.exists():
        report_rows.append({"report": name, "status": "missing", "failed_checks": "missing", "next": ""})
        continue
    report = read_json(path)
    checks = report.get("checks", {})
    failed = [key for key, value in checks.items() if value is False]
    report_rows.append({
        "report": name,
        "status": report.get("status", ""),
        "failed_checks": ",".join(failed) if failed else "-",
        "next": report.get("next_required_step", ""),
    })

print_table(report_rows, ["report", "status", "failed_checks", "next"])

causal_report = read_json(report_paths["causal_shortest_path"])
geo_report = read_json(report_paths["geoattention_v2_mechanics"])
print("\ncausal_future_edges_forbidden=", causal_report.get("checks", {}).get("future_edges_forbidden"))
print("geoattention_future_attention_forbidden=", geo_report.get("checks", {}).get("future_attention_forbidden"))
print("geoattention_strict_gate_status=", geo_report.get("strict_gate_status"))

pretraining_required_checks = {
    "all_reports_present": all(Path(path).exists() for path in report_paths.values()),
    "strict_controls_status_pass": read_json(report_paths["strict_w_controls"]).get("status") == "pass",
    "memory_observer_status_pass": read_json(report_paths["relational_memory"]).get("status") == "pass",
    "causal_geometry_status_pass": causal_report.get("status") == "pass",
    "causal_future_edges_forbidden": causal_report.get("checks", {}).get("future_edges_forbidden") is True,
    "causal_future_inputs_forbidden": causal_report.get("checks", {}).get("future_inputs_forbidden") is True,
    "geoattention_mechanics_status_pass": geo_report.get("status") == "pass",
    "geoattention_required_controls_present": geo_report.get("checks", {}).get("required_controls_present") is True,
    "geoattention_stable_causal_geometry_injected": geo_report.get("checks", {}).get("stable_causal_geometry_injected") is True,
    "geoattention_future_attention_forbidden": geo_report.get("checks", {}).get("future_attention_forbidden") is True,
}

observer_failures = []
for row in report_rows:
    if row["status"] != "pass":
        observer_failures.append(f"{row['report']}: status={row['status']}")
    if row["failed_checks"] not in {"-", "", "missing"}:
        observer_failures.append(f"{row['report']}: failed_checks={row['failed_checks']}")
for check_name, value in pretraining_required_checks.items():
    if value is not True:
        observer_failures.append(f"{check_name}={value}")

pretraining_gate = {
    "status": "pass" if not observer_failures else "failed",
    "checks": pretraining_required_checks,
    "failures": observer_failures,
    "note": "geoattention strict training gate is expected to require the training controls below",
}
write_json(RUN_ROOT / "pretraining_gate_summary.json", pretraining_gate)
print("\npretraining_gate_status=", pretraining_gate["status"])
if observer_failures:
    print("pretraining_failures:")
    for failure in observer_failures:
        print("-", failure)
    if EXPORT_REPORT_BUNDLE:
        export_report_bundle(reason="pretraining_gate_failed")
    if ABORT_ON_FAILED_OBSERVER_GATE:
        raise RuntimeError("Pre-training observer gate failed; training controls were not started.")

## 5. Build Short Training Control Configs

کنترل های این نوت بوک:

- `baseline`
- `alpha_zero`
- `real_memory_d`
- `random_memory_d`
- `shuffled_memory_d`
- `no_memory_real_d`
- `instantaneous_real_d` اگر `INCLUDE_INSTANTANEOUS_CONTROL=True` باشد

همه configها در `RUN_ROOT/configs` ساخته می شوند و خروجی train در همان run folder ذخیره می شود.

In [ ]:
BASELINE_TEMPLATE = read_json("configs/baseline/phase3_stable_base_seed2027.json")
ERGT_TEMPLATE = read_json("configs/ergt_v1/phase3_stable_base/real_d_alpha_0_1_warmup_cosine_seed2027.json")


def apply_common_budget(config, condition, output_dir):
    config["run"]["phase"] = "notebook_01_attention_evidence_ladder"
    config["run"]["condition"] = condition
    config["run"]["seed"] = SEED
    config["run"]["output_dir"] = Path(output_dir).as_posix()
    config["dataset"]["context_length"] = 256
    config["model"]["n_layers"] = BUDGET["n_layers"]
    config["model"]["n_heads"] = BUDGET["n_heads"]
    config["model"]["hidden_dim"] = BUDGET["hidden_dim"]
    config["model"]["ffn_dim"] = BUDGET["ffn_dim"]
    config["training"]["batch_size"] = BUDGET["batch_size"]
    config["training"]["max_steps"] = BUDGET["max_steps"]
    config["training"]["eval_interval"] = BUDGET["eval_interval"]
    config["training"]["checkpoint_interval"] = BUDGET["max_steps"]
    config["training"]["warmup_steps"] = min(BUDGET["warmup_steps"], BUDGET["max_steps"])
    return config


def make_baseline_config(condition):
    output_dir = RUN_ROOT / condition
    config = apply_common_budget(copy.deepcopy(BASELINE_TEMPLATE), condition, output_dir)
    return config


def make_ergt_config(condition, distance_mode, alpha):
    output_dir = RUN_ROOT / condition
    config = apply_common_budget(copy.deepcopy(ERGT_TEMPLATE), condition, output_dir)
    attention = config.setdefault("attention", {})
    attention["distance_mode"] = distance_mode
    attention["causal_runtime_distance"] = True
    attention["max_causal_step"] = 1
    attention["gradient_mode"] = "detached_d"
    attention["memory"] = {
        "decay": 0.7,
        "eta": 0.3,
        "gate_floor": 0.05,
        "min_context_edges": 2,
    }
    attention["alpha"] = {
        "mode": "fixed",
        "initial_value": alpha,
        "non_negative": True,
        "warmup_steps": min(BUDGET["warmup_steps"], BUDGET["max_steps"]),
    }
    config.setdefault("logging", {})["log_geometry_diagnostics"] = True
    return config


control_specs = [
    {"condition": "baseline", "kind": "baseline"},
    {"condition": "alpha_zero", "kind": "ergt", "distance_mode": "real_stable_causal_d", "alpha": 0.0},
    {"condition": "real_memory_d", "kind": "ergt", "distance_mode": "real_stable_causal_d", "alpha": ALPHA},
    {"condition": "random_memory_d", "kind": "ergt", "distance_mode": "random_stable_causal_d", "alpha": ALPHA},
    {"condition": "shuffled_memory_d", "kind": "ergt", "distance_mode": "shuffled_stable_causal_d", "alpha": ALPHA},
    {"condition": "no_memory_real_d", "kind": "ergt", "distance_mode": "no_memory_real_d", "alpha": ALPHA},
]
if INCLUDE_INSTANTANEOUS_CONTROL:
    control_specs.append({
        "condition": "instantaneous_real_d",
        "kind": "ergt",
        "distance_mode": "instantaneous_real_d",
        "alpha": ALPHA,
    })

run_plan = []
for spec in control_specs:
    condition = spec["condition"]
    if spec["kind"] == "baseline":
        config = make_baseline_config(condition)
        script = "experiments/train_baseline.py"
    else:
        config = make_ergt_config(condition, spec["distance_mode"], spec["alpha"])
        script = "experiments/train_ergt_v1.py"
    config_path = CONFIG_DIR / f"{condition}.json"
    write_json(config_path, config)
    run_plan.append({
        **spec,
        "config_path": config_path,
        "output_dir": Path(config["run"]["output_dir"]),
        "script": script,
    })

print_table([
    {
        "condition": item["condition"],
        "kind": item["kind"],
        "distance_mode": item.get("distance_mode", "-"),
        "alpha": item.get("alpha", "-"),
        "config": item["config_path"].as_posix(),
    }
    for item in run_plan
], ["condition", "kind", "distance_mode", "alpha", "config"])

## 6. Run Training Controls

این مرحله ممکن است زمان بر باشد. برای profile اول، `smoke` را نگه دار. اگر خطا نداد، نوت بوک را با `RUN_PROFILE="sanity"` و بعد `RUN_PROFILE="evidence"` دوباره اجرا کن.

In [ ]:
training_records = []
per_run_timeout_sec = int(BUDGET["per_run_timeout_minutes"] * 60)

if RUN_TRAINING and DATA_READY:
    for item in run_plan:
        cmd = [
            sys.executable,
            item["script"],
            "--config",
            item["config_path"],
            "--device",
            DEVICE,
        ]
        result = run_cmd(cmd, timeout_sec=per_run_timeout_sec, check=STOP_ON_FIRST_FAILURE)
        training_records.append({
            "condition": item["condition"],
            "returncode": result["returncode"],
            "elapsed_seconds": result["elapsed_seconds"],
        })
else:
    print("Training skipped.")
    print(f"RUN_TRAINING={RUN_TRAINING}, DATA_READY={DATA_READY}")

print_table(training_records, ["condition", "returncode", "elapsed_seconds"])

## 7. Collect Metrics

In [ ]:
def load_training_metrics(item):
    output_dir = Path(item["output_dir"])
    metrics_path = output_dir / ("baseline_results.json" if item["kind"] == "baseline" else "metrics.json")
    if metrics_path.exists():
        metrics = read_json(metrics_path)
        source = metrics_path.as_posix()
    else:
        progress = read_jsonl(output_dir / "progress_log.jsonl")
        validation_records = [row for row in progress if "validation_loss" in row]
        metrics = validation_records[-1] if validation_records else {}
        source = "progress_log_fallback" if metrics else "missing"

    geometry_summary = metrics.get("geometry", {}).get("summary", {}) if isinstance(metrics.get("geometry"), dict) else {}
    return {
        "condition": item["condition"],
        "kind": item["kind"],
        "distance_mode": item.get("distance_mode", "-"),
        "best_validation_loss": finite_or_none(metrics.get("best_validation_loss", metrics.get("validation_loss"))),
        "final_validation_loss": finite_or_none(metrics.get("final_validation_loss", metrics.get("validation_loss"))),
        "perplexity": finite_or_none(metrics.get("perplexity")),
        "geo_to_qk_ratio": finite_or_none(geometry_summary.get("geo_to_qk_ratio")),
        "attention_entropy": finite_or_none(geometry_summary.get("attention_entropy")),
        "tokens_per_second": finite_or_none(metrics.get("average_tokens_per_second")),
        "source": source,
    }


metric_rows = [load_training_metrics(item) for item in run_plan]
print_table(metric_rows, [
    "condition",
    "distance_mode",
    "best_validation_loss",
    "final_validation_loss",
    "geo_to_qk_ratio",
    "attention_entropy",
    "source",
])

## 8. Attention Evidence Gate

این gate برای اجرای کوتاه، حکم اثبات نهایی ندارد. اگر smoke/sanity پاس شود اما اختلاف ها کوچک یا ناپایدار باشند، خروجی درست `needs_longer_run` است. pass واقعی را باید با evidence و چند seed تثبیت کرد.

In [ ]:
scores = {
    row["condition"]: row["best_validation_loss"]
    for row in metric_rows
    if row["best_validation_loss"] is not None
}


def lower_is_better(left, right):
    if left not in scores or right not in scores:
        return None
    return scores[left] < scores[right]


gate_checks = {
    "observer_reports_present": all(Path(path).exists() for path in report_paths.values()),
    "strict_controls_pass": read_json(report_paths["strict_w_controls"]).get("status") == "pass",
    "causal_future_edges_forbidden": read_json(report_paths["causal_shortest_path"]).get("checks", {}).get("future_edges_forbidden") is True,
    "attention_future_forbidden": read_json(report_paths["geoattention_v2_mechanics"]).get("checks", {}).get("future_attention_forbidden") is True,
    "real_beats_baseline": lower_is_better("real_memory_d", "baseline"),
    "real_beats_alpha_zero": lower_is_better("real_memory_d", "alpha_zero"),
    "real_beats_random": lower_is_better("real_memory_d", "random_memory_d"),
    "real_beats_shuffled": lower_is_better("real_memory_d", "shuffled_memory_d"),
    "real_beats_no_memory": lower_is_better("real_memory_d", "no_memory_real_d"),
}
if INCLUDE_INSTANTANEOUS_CONTROL:
    gate_checks["real_beats_instantaneous"] = lower_is_better("real_memory_d", "instantaneous_real_d")

if not RUN_TRAINING or not DATA_READY:
    gate_status = "not_run_training_controls"
elif any(value is False for value in gate_checks.values()):
    gate_status = "needs_investigation"
elif any(value is None for value in gate_checks.values()):
    gate_status = "incomplete"
elif RUN_PROFILE in {"smoke", "sanity"}:
    gate_status = "mechanics_pass_needs_evidence_run"
else:
    gate_status = "attention_evidence_pass"

gate_rows = [{"check": key, "value": value} for key, value in gate_checks.items()]
print_table(gate_rows, ["check", "value"])
print(f"\ngate_status={gate_status}")

attention_gate_failures = [
    f"{key}={value}" for key, value in gate_checks.items() if value is not True
]

summary = {
    "run_profile": RUN_PROFILE,
    "run_root": RUN_ROOT.as_posix(),
    "budget": BUDGET,
    "device": DEVICE,
    "alpha": ALPHA,
    "gate_status": gate_status,
    "gate_checks": gate_checks,
    "gate_failures": attention_gate_failures,
    "metrics": metric_rows,
}
summary_path = RUN_ROOT / "attention_evidence_summary.json"
write_json(summary_path, summary)
print(f"summary_path={summary_path}")

if EXPORT_REPORT_BUNDLE:
    export_report_bundle(reason=gate_status)

if gate_status in {"needs_investigation", "incomplete", "not_run_training_controls"}:
    if attention_gate_failures:
        print("attention_gate_failures:")
        for failure in attention_gate_failures:
            print("-", failure)
    if ABORT_ON_FAILED_ATTENTION_GATE:
        raise RuntimeError(f"Attention evidence gate failed: {gate_status}")

## 9. Decision Rule

- اگر `gate_status=mechanics_pass_needs_evidence_run` شد، نوت بوک را با `RUN_PROFILE="evidence"` اجرا کن.
- اگر `gate_status=attention_evidence_pass` شد، مرحله بعد اجرای بلند چند seed و سپس باز کردن Auxiliary Physics-Inspired Loss است.
- اگر `gate_status=needs_investigation` شد، قبل از اجرای بلند باید alpha، normalization، memory decay/eta، یا fairness کنترل ها بررسی شود.
- اگر training skip شد، ابتدا دیتای WikiText-2 را آماده کن و smoke را دوباره اجرا کن.

این نوت بوک برای رد یا قبول سریع مسیر attention است؛ نه برای claim نهایی معماری کامل.

در پایان اجرا، فقط فایل های لازم برای بررسی بیرونی zip می شوند و checkpointها داخل zip نمی آیند. نام فایل خروجی ثابت است:

`ergt_01_attention_evidence_report_bundle.zip`

وقتی این فایل از Colab دانلود شود، مسیر پیش فرض خواندن آن در ماشین محلی این است:

`C:\\Users\\Administrator\\Downloads\\ergt_01_attention_evidence_report_bundle.zip`